# 04 — Complete-game AHR self-play and joint training

This notebook starts the fresh Scout/Worker system learning from its own complete 9×9 Go games. PUCT's **real completed visits** supervise the Scout; verified two-pass terminal games supervise current-player loss/draw/win, score and ownership. There is no tactical threshold to clear before game learning.

**Inputs:** frozen 00–03 tagged definitions; a JSON run configuration; GPU 1 for the authorized long run. **Outputs:** resumable `checkpoints/<run_id>/`, per-event `progress.json` / `reports/ahr_v2/<run_id>/status.json`, game/training metrics, and a read-only player loader for 05 and 06. **Runtime:** Run All executes disposable CPU and GPU recovery checks (typically several minutes), not the 12-hour run. Notebook 07 starts the authorized long run in detached tmux.

**Claims:** self-play targets are search estimates; terminal labels are verified under the configured area/positional-superko rules. Neither predicts a proved winning strategy. Max-ply games remain UNKNOWN and have policy labels only. No KataGo or GNU Go teaching labels are used.

Whole games are assigned to train/dev/lockbox **before** D4 augmentation. Dev and lockbox never enter the optimizer. Captures must be played out; no dead-stone oracle or forced terminal pass is supplied.

In [ ]:
from pathlib import Path
import os, nbformat
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT: raise RuntimeError('Open in the AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_v2/00_contracts_and_recovery.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(cell.source,globals())
for name in GO2_NOTEBOOKS[1:]: go2_import_exports(PROJECT/GO2_ROOT/name,globals())
print('Imported definitions only from 00–03; no upstream training cells executed.')

## Configuration and source identity
The configured device is `cuda:1`; the unrelated GPU 0 workload stays independent. Search uses 32 visits initially, then 64 and 128 after actual completed games. All legal actions, including pass, remain eligible. A game yields optimizer updates as soon as it finishes.

In [ ]:
import collections, math
GO2_TRAINING_NOTEBOOK='04_selfplay_and_training.ipynb'

def go2_training_config(run_id='ahr-v2-selfplay-20261006', device='cuda:1', time_budget_hours=12):
    return {'schema':'ahr-v2-selfplay-config-v1','run_id':str(run_id),'seed':2026100604,
        'device':str(device),'board_size':9,'komi':7.5,'network':{'width':64,'blocks':4},
        'time_budget_hours':float(time_budget_hours),'max_completed_games':None,'max_total_games':None,
        'max_plies':729,'search_batch_size':8,'search_schedule':[[0,32],[32,64],[128,128]],
        'root_noise_fraction':0.25,'root_dirichlet_alpha':0.15,'temperature_moves':60,
        'early_temperature':1.0,'late_temperature':0.35,'batch_size':64,'updates_per_game':32,
        'unknown_updates':4,'learning_rate':0.001,'weight_decay':0.0001,'gradient_clip':5.0,
        'replay_train_games':128,'replay_dev_games':64,'replay_lockbox_games':64,
        'augmentation':True,'checkpoint_seconds':120,'checkpoint_optimizer_steps':64,
        'cpu_threads':4,'is_smoke':False,'training_source':'ahr_selfplay',
        'loss':{'policy_weight':1.0,'outcome_weight':1.0,'score_weight':0.25,'ownership_weight':0.25,'l2_weight':0.0}}

def go2_validate_training_config(config):
    base=go2_training_config(); unknown=set(config)-set(base)
    if unknown: raise ValueError('Unknown training settings: '+str(sorted(unknown)))
    config={**base,**copy.deepcopy(config)}
    if config['schema']!=base['schema'] or config['training_source']!='ahr_selfplay':raise ValueError('Only pure AHR self-play is implemented')
    if Path(config['run_id']).name!=config['run_id'] or not config['run_id']:raise ValueError('run_id must be a local directory name')
    if config['device'] not in ('cpu','cuda:1'):raise ValueError('Use CPU or GPU 1; GPU 0 is reserved for the independent workload')
    for key in ('board_size','max_plies','search_batch_size','temperature_moves','batch_size','updates_per_game','replay_train_games','replay_dev_games','replay_lockbox_games','cpu_threads'):
        if type(config[key]) is not int or config[key]<1:raise ValueError('Positive integer required: '+key)
    if config['checkpoint_optimizer_steps']>250 or config['checkpoint_seconds']>900:raise ValueError('Checkpoint cadence exceeds the workspace agreement')
    if config['time_budget_hours']<=0:raise ValueError('Positive measured active-time budget required')
    if config['max_plies']<2 or not config['search_schedule'] or config['search_schedule'][0][0]!=0:raise ValueError('Invalid game/search schedule')
    if any(len(item)!=2 or item[0]<0 or item[1]<1 for item in config['search_schedule']):raise ValueError('Invalid search schedule entries')
    if sorted(config['search_schedule'])!=config['search_schedule']:raise ValueError('Search schedule must increase by completed-game count')
    Go2NetConfig(board_size=config['board_size'],**config['network']);Go2LossConfig(**config['loss'])
    return config

def go2_training_sources(project):
    sources=go2_source_identity(project)
    sources[GO2_ROOT+'/'+GO2_TRAINING_NOTEBOOK]=go2_code_sha(Path(project)/GO2_ROOT/GO2_TRAINING_NOTEBOOK)
    return sources

def go2_training_identity(project,config):
    return {'schema':GO2_SCHEMA,'sources':go2_training_sources(project),'config':copy.deepcopy(config),
        'torch':torch.__version__,'numpy':np.__version__,'rules':GO2_RULE_PROFILE}

def go2_training_determinism(config):
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
    torch.set_num_threads(config['cpu_threads']);torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    if config['device'].startswith('cuda') and (not torch.cuda.is_available() or torch.cuda.device_count()<2):
        raise RuntimeError('GPU 1 is unavailable; refusing silent CPU/device fallback')

def go2_search_budget(config,completed_games):
    return int([visits for count,visits in config['search_schedule'] if count<=completed_games][-1])

def go2_game_split(seed,game_index):
    if game_index==0:return 'train'
    value=int(go2_digest(['whole_game_split',int(seed),int(game_index)])[:8],16)%10
    return 'dev' if value==8 else 'lockbox' if value==9 else 'train'


## Replay and labels
Replay stores whole game action lists and search visits, not duplicated game histories. Position caches are rebuilt deterministically after resume. UNKNOWN records supply policy estimates only. Symmetry is drawn from the saved private sampler after selecting a training game and position.

In [ ]:
def go2_game_cache(game,cache):
    key=game['game_id']
    if key not in cache:
        state=go2_initial_state(game['size'],game['komi']);states=[]
        for action in game['actions']:
            states.append(state);state=go2_step(state,int(action))
        if go2_state_digest(state)!=game['final_state_sha256']:raise RuntimeError('Replay game final digest mismatch')
        if (game['status']=='TERMINAL')!=state.terminal:raise RuntimeError('Replay terminal status mismatch')
        cache[key]=(states,state)
    return cache[key]

def go2_replay_row(game,index,symmetry,cache):
    states,terminal=go2_game_cache(game,cache);state=states[index]
    policy=game['positions'][index]['policy'];search=game['positions'][index]
    if symmetry:
        state=go2_transform_state(state,symmetry)
        policy=go2_transform_vector(policy,state.size,symmetry)
    provenance={'source':'ahr_puct_completed_visits','source_id':game['game_id']+':'+str(index),
        'perspective':'action','certainty':'search_estimate','completed_simulations':search['visits'],
        'network_sha256':search['model_identity'],'game_family':game['game_id'],'d4':int(symmetry)}
    labels={'policy':{'target':list(map(float,policy)),'provenance':provenance}}
    if game['status']=='TERMINAL':
        transformed_terminal=go2_transform_state(terminal,symmetry) if symmetry else terminal
        score=go2_score(transformed_terminal);outcome=go2_outcome(transformed_terminal)
        value=state.to_play*int(outcome['value_black'])
        exact={'source':'exact_terminal_replay','source_id':game['game_id'],'perspective':'current_player',
            'certainty':'terminal_verified','terminal_state_sha256':go2_state_digest(transformed_terminal),
            'position_state_sha256':go2_state_digest(state),'game_family':game['game_id'],'d4':int(symmetry)}
        labels['outcome']={'target':[float(value==-1),float(value==0),float(value==1)],'provenance':dict(exact)}
        labels['score']={'target':state.to_play*score['score_black']/(state.size*state.size+abs(state.komi)),'provenance':dict(exact)}
        labels['ownership']={'target':(state.to_play*np.asarray(score['ownership_black'],dtype=np.float32)).tolist(),'provenance':dict(exact)}
    return {'state':go2_state_to_dict(state),'labels':labels}

def go2_sample_training_rows(state,config,generator,cache):
    games=state['replay']['train']
    eligible=[game for game in games if game['positions']]
    if not eligible:raise RuntimeError('No training rows; never fall back to dev or lockbox')
    rows=[]
    for _ in range(config['batch_size']):
        gi=int(torch.randint(len(eligible),(1,),generator=generator).item());game=eligible[gi]
        index=int(torch.randint(len(game['positions']),(1,),generator=generator).item())
        symmetry=int(torch.randint(8,(1,),generator=generator).item()) if config['augmentation'] else 0
        rows.append(go2_replay_row(game,index,symmetry,cache))
        state['sampler']['rows_seen']+=1
    return rows

def go2_finish_training_game(state,config,cache):
    active=state['self_play'];position=go2_state_from_dict(active['state'])
    status='TERMINAL' if position.terminal else 'UNKNOWN'
    game={'game_id':active['game_id'],'game_index':active['game_index'],'family':active['game_id'],
        'split':active['split'],'size':config['board_size'],'komi':config['komi'],
        'actions':list(position.actions),'positions':active['positions'],'status':status,
        'final_state_sha256':go2_state_digest(position),'outcome':go2_outcome(position) if position.terminal else None,
        'cutoff_reason':None if position.terminal else 'max_plies'}
    # Every stored row corresponds to a real search at a game prefix.
    if len(game['actions'])!=len(game['positions']):raise RuntimeError('Search/game position counts differ')
    go2_game_cache(game,cache)
    bucket=game['split'];state['replay'][bucket].append(game)
    retain=config['replay_'+bucket+'_games']
    dropped=state['replay'][bucket][:-retain]
    state['replay'][bucket]=state['replay'][bucket][-retain:]
    for old in dropped:cache.pop(old['game_id'],None)
    metrics=state['metrics'];metrics['total_games']+=1
    metrics['completed_games' if position.terminal else 'unknown_games']+=1
    metrics['games_by_split'][bucket]+=1;metrics['plies']+=len(game['actions'])
    metrics['recent_games']=(metrics['recent_games']+[{'game_id':game['game_id'],'status':status,'split':bucket,
        'plies':len(game['actions']),'score_black':game['outcome']['score_black'] if game['outcome'] and bucket!='lockbox' else None}])[-64:]
    state['self_play']=None;state['search']=None
    state['updates_remaining']=(config['updates_per_game'] if position.terminal else config['unknown_updates']) if bucket=='train' else 0
    state['stage']='training' if state['updates_remaining'] else 'new_game'
    return game

def go2_training_progress(state,entry=None,status='RUNNING'):
    config=state['config'];metrics=state['metrics'];active=state['self_play']
    return {'schema':'ahr-v2-training-progress-v1','run_id':config['run_id'],'status':status,'stage':state['stage'],
        'event_step':state['global_step'],'global_step':state['global_step'],'optimizer_step':metrics['optimizer_step'],
        'completed_games':metrics['completed_games'],'unknown_games':metrics['unknown_games'],'total_games':metrics['total_games'],
        'replay_train_games':len(state['replay']['train']),'replay_train_rows':sum(len(g['positions']) for g in state['replay']['train']),
        'replay_dev_games':len(state['replay']['dev']),'replay_lockbox_games':len(state['replay']['lockbox']),
        'current_game':None if active is None else active['game_id'],
        'current_ply':None if active is None else len(active['positions']),'last_loss':metrics['last_loss'],
        'elapsed_seconds':state['active_seconds'],'remaining_seconds':max(0,config['time_budget_hours']*3600-state['active_seconds']),
        'latest_checkpoint':None if entry is None else entry['file'],'sources':state['identity']['sources'],
        'training_source':'ahr_selfplay','is_smoke':config['is_smoke'],'updated_at_utc':go2_now()}


## Resumable event loop
A bounded event is one search batch, one new game, one completed-game handoff, or one optimizer step. A paused run saves the active tree, game, replay, sampler and all RNG. Event count is the checkpoint cursor; `metrics.optimizer_step` is the actual number of updates. Best means the lowest smoothed training loss among saved snapshots, **not** strongest measured player.

In [ ]:
def go2_run_training(project=None,config=None,max_events=None,on_checkpoint=None):
    project=go2_project(project);config=go2_validate_training_config(config or go2_training_config())
    go2_training_determinism(config);directory=project/'checkpoints'/config['run_id']
    identity=go2_training_identity(project,config);report_dir=project/'reports/ahr_v2'/config['run_id']
    report_dir.mkdir(parents=True,exist_ok=True);cache={}
    with go2_checkpoint_lock(directory):
        saved,entry=go2_load_checkpoint(directory,identity)
        go2_seed(config['seed'])
        model=Go2Network(Go2NetConfig(board_size=config['board_size'],**config['network'])).to(config['device'])
        optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=config['weight_decay'])
        scheduler=torch.optim.lr_scheduler.ConstantLR(optimizer,factor=1.0,total_iters=1)
        scaler=torch.amp.GradScaler('cuda',enabled=False)
        sampler=torch.Generator(device='cpu').manual_seed(config['seed']+101)
        action_rng=np.random.default_rng(config['seed']+202)
        state={'schema':GO2_SCHEMA,'identity':identity,'config':config,'global_step':0,'stage':'new_game',
            'model':{},'optimizer':{},'scheduler':{},'scaler':{},'rng':{},
            'sampler':{'rows_seen':0,'action_rng':copy.deepcopy(action_rng.bit_generator.state)},
            'replay':{'train':[],'dev':[],'lockbox':[]},'self_play':None,'search':None,'updates_remaining':0,
            'active_seconds':0.0,'metrics':{'optimizer_step':0,'completed_games':0,'unknown_games':0,'total_games':0,
                'games_by_split':{'train':0,'dev':0,'lockbox':0},'plies':0,'last_loss':None,'loss_ema':None,
                'best_loss_ema':None,'recent_games':[],'recent_losses':[],'first_complete_game_seconds':None}}
        if saved is not None:
            state=saved;model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
            go2_restore_rng(state['rng'],{'sampler':sampler});action_rng.bit_generator.state=copy.deepcopy(state['sampler']['action_rng'])
            go2_invalidate_model_identity(model)
        go2_atomic_json(directory/'config.json',config)
        last_saved=-1 if entry is None else state['global_step'];last_checkpoint=time.monotonic();steps_saved=state['metrics']['optimizer_step']
        processed=0;status='RUNNING';live_tree=None
        def save_snapshot(force=False):
            nonlocal entry,last_saved,last_checkpoint,steps_saved
            if state['global_step']==last_saved:return
            if not force and time.monotonic()-last_checkpoint<config['checkpoint_seconds'] and state['metrics']['optimizer_step']-steps_saved<config['checkpoint_optimizer_steps']:return
            state['sampler']['action_rng']=copy.deepcopy(action_rng.bit_generator.state)
            state.update(model=copy.deepcopy(model.state_dict()),optimizer=copy.deepcopy(optimizer.state_dict()),
                scheduler=copy.deepcopy(scheduler.state_dict()),scaler=copy.deepcopy(scaler.state_dict()),
                rng=go2_capture_rng(cuda_devices=(1,) if config['device']=='cuda:1' else (),generators={'sampler':sampler}))
            best=False;ema=state['metrics']['loss_ema'];old=state['metrics']['best_loss_ema']
            if ema is not None and (old is None or ema<old):state['metrics']['best_loss_ema']=ema;best=True
            entry=go2_save_checkpoint(directory,state,best=best,
                resume_cell='04/07: go2_run_training(PROJECT, json.loads((PROJECT/"checkpoints"/run_id/"config.json").read_text()))')
            last_saved=state['global_step'];last_checkpoint=time.monotonic();steps_saved=state['metrics']['optimizer_step']
            if on_checkpoint is not None:
                # Observers may inspect immutable snapshots; protect training RNG from diagnostic work.
                rng=go2_capture_rng(cuda_devices=(1,) if config['device']=='cuda:1' else (),generators={'sampler':sampler})
                try:on_checkpoint(state,entry)
                finally:go2_restore_rng(rng,{'sampler':sampler})
        def write_progress():
            progress=go2_training_progress(state,entry,status)
            go2_atomic_json(directory/'progress.json',progress);go2_atomic_json(report_dir/'status.json',progress)
        save_snapshot(True);write_progress()
        try:
            while True:
                if (directory/'STOP').exists():status='STOPPED';break
                if state['active_seconds']>=config['time_budget_hours']*3600:status='TIME_BUDGET_COMPLETE';break
                if config['max_completed_games'] is not None and state['metrics']['completed_games']>=config['max_completed_games'] and state['stage']!='training':status='GAME_LIMIT_COMPLETE';break
                if config['max_total_games'] is not None and state['metrics']['total_games']>=config['max_total_games'] and state['stage']!='training':status='GAME_LIMIT_COMPLETE';break
                if max_events is not None and processed>=int(max_events):status='PAUSED';break
                started=time.monotonic();previous_stage=state['stage'];logged=None
                if state['stage']=='new_game':
                    gi=state['metrics']['total_games'];game_id='ahr-selfplay-'+str(config['seed'])+'-'+str(gi)
                    state['self_play']={'game_id':game_id,'game_index':gi,'split':go2_game_split(config['seed'],gi),
                        'state':go2_state_to_dict(go2_initial_state(config['board_size'],config['komi'])),'positions':[]}
                    state['stage']='selfplay'
                elif state['stage']=='selfplay':
                    active=state['self_play'];position=go2_state_from_dict(active['state'])
                    if position.terminal or len(position.actions)>=config['max_plies']:
                        finished=go2_finish_training_game(state,config,cache)
                        logged={key:finished[key] for key in ('game_id','game_index','split','status','cutoff_reason')}
                        logged['plies']=len(finished['actions'])
                        logged['score_black']=finished['outcome']['score_black'] if finished['outcome'] and finished['split']!='lockbox' else None
                        if position.terminal and state['metrics']['first_complete_game_seconds'] is None:
                            state['metrics']['first_complete_game_seconds']=state['active_seconds']
                    else:
                        if state['search'] is None:
                            search_config=Go2SearchConfig(simulations=go2_search_budget(config,state['metrics']['completed_games']),
                                batch_size=config['search_batch_size'],seed=int(action_rng.integers(0,2**31-1)),
                                root_noise_fraction=config['root_noise_fraction'],root_dirichlet_alpha=config['root_dirichlet_alpha'])
                            tree=None
                        else:
                            search_config=Go2SearchConfig(**state['search']['config'])
                            tree=live_tree if live_tree is not None else go2_search_from_state_dict(state['search'],model,position,search_config,
                                context={'game':active['game_id'],'ply':len(position.actions)},project=project)
                        completed=0 if tree is None else tree['completed_simulations']
                        result=go2_search(model,position,search_config,tree=tree,
                            stop_after_simulations=min(search_config.simulations,completed+search_config.batch_size),
                            context={'game':active['game_id'],'ply':len(position.actions)},project=project)
                        live_tree=result['tree'];state['search']=go2_search_state_dict(live_tree)
                        if result['status']=='COMPLETE':
                            if not result['policy_target_valid']:raise RuntimeError('No real visit target; refusing invented policy labels')
                            policy=np.asarray(result['policy_target'],dtype=np.float64)
                            temperature=config['early_temperature'] if len(position.actions)<config['temperature_moves'] else config['late_temperature']
                            if temperature<=0:action=int(result['action'])
                            else:
                                probability=np.power(policy,1/temperature);probability/=probability.sum()
                                action=int(action_rng.choice(len(probability),p=probability))
                            active['positions'].append({'policy':policy.tolist(),'visits':result['completed_simulations'],
                                'model_identity':go2_model_identity(model),'action':action})
                            active['state']=go2_state_to_dict(go2_step(position,action));state['search']=None;live_tree=None
                elif state['stage']=='training':
                    model.train();rows=go2_sample_training_rows(state,config,sampler,cache)
                    optimizer.zero_grad(set_to_none=True);loss=go2_joint_loss(model,rows,Go2LossConfig(**config['loss']))
                    loss['loss'].backward();torch.nn.utils.clip_grad_norm_(model.parameters(),config['gradient_clip'],error_if_nonfinite=True)
                    optimizer.step();scheduler.step();go2_invalidate_model_identity(model)
                    metrics=state['metrics'];metrics['optimizer_step']+=1;metrics['last_loss']=float(loss['loss'].detach().cpu())
                    metrics['loss_ema']=metrics['last_loss'] if metrics['loss_ema'] is None else 0.9*metrics['loss_ema']+0.1*metrics['last_loss']
                    record={'optimizer_step':metrics['optimizer_step'],'loss':metrics['last_loss'],
                        'components':{k:float(v.detach().cpu()) for k,v in loss['components'].items()},'label_counts':loss['counts']}
                    metrics['recent_losses']=(metrics['recent_losses']+[record])[-256:];logged=record
                    state['updates_remaining']-=1
                    if state['updates_remaining']==0:state['stage']='new_game'
                else:raise RuntimeError('Unknown saved stage; refusing silent restart')
                state['global_step']+=1;processed+=1;state['active_seconds']+=time.monotonic()-started
                if logged is not None:
                    with (report_dir/'metrics.jsonl').open('a') as stream:
                        stream.write(json.dumps({'event_step':state['global_step'],'active_seconds':state['active_seconds'],
                            'kind':'game' if 'game_id' in logged else 'optimizer','data':logged},default=go2_json_default)+'\n');stream.flush()
                save_snapshot(previous_stage!=state['stage']);write_progress()
        except BaseException as error:
            # A failed event may have advanced RNG or partially changed optimizer state.
            # Preserve the last valid manifest and replay this event on explicit resume.
            try:
                valid,valid_entry=go2_load_checkpoint(directory,identity)
                failed=go2_training_progress(valid,valid_entry,'ERROR') if valid is not None else {'run_id':config['run_id'],'status':'ERROR','event_step':last_saved}
            except Exception as checkpoint_error:
                failed={'run_id':config['run_id'],'status':'ERROR','event_step':last_saved,'checkpoint_validation_error':repr(checkpoint_error)}
            failed.update(error=repr(error),discarded_partial_event=True,updated_at_utc=go2_now())
            go2_atomic_json(directory/'progress.json',failed);go2_atomic_json(report_dir/'status.json',failed)
            raise
        save_snapshot(True);write_progress()
        return state

def go2_load_player(project=None,run_id=None,which='latest',device='cpu'):
    project=go2_project(project)
    if run_id is None:
        active=project/'reports/ahr_v2/active_run.json'
        config_path=project/'config/ahr_v2_training.json'
        if active.exists():run_id=json.loads(active.read_text())['run_id']
        elif config_path.exists():run_id=json.loads(config_path.read_text())['run_id']
        else:raise FileNotFoundError('No active AHR v2 run; choose an explicit run_id')
    directory=project/'checkpoints'/str(run_id)
    for attempt in range(4):
        before=(directory/'manifest.json').read_bytes()
        try:state,entry=go2_load_checkpoint(directory,which=which)
        except (RuntimeError,FileNotFoundError):
            after=(directory/'manifest.json').read_bytes()
            if before==after or attempt==3:raise
            continue
        if state is None:raise FileNotFoundError('No playable checkpoint yet')
        break
    sources=go2_training_sources(project)
    if state['identity']['sources']!=sources:raise RuntimeError('Player checkpoint sources changed; refusing mismatched inference')
    config=state['config']
    with torch.random.fork_rng(devices=[]):
        model=Go2Network(Go2NetConfig(board_size=config['board_size'],**config['network']))
    model.load_state_dict(state['model']);model.to(device);model.eval();model.requires_grad_(False)
    metadata={'run_id':str(run_id),'global_step':state['global_step'],'optimizer_step':state['metrics']['optimizer_step'],
        'model_identity':go2_model_identity(model),'checkpoint_file':entry['file'],'checkpoint_sha256':entry['sha256'],
        'source_identity':state['identity'],'sources':sources,'config':config,'training_source':'ahr_selfplay',
        'is_smoke':config['is_smoke'],'best_selection_basis':'lowest_saved_training_loss_ema_not_playing_strength','completed_games':state['metrics']['completed_games'],'unknown_games':state['metrics']['unknown_games']}
    return model,metadata


## Recovery checks
The CPU fixture plays real searched games on a disposable 3×3 board and performs joint optimizer updates. It stops inside an active search, resumes in a fresh Jupyter kernel, and compares model, optimizer, scheduler, sampler, replay, game/search state, metrics and every saved RNG. Wall-clock timestamps/durations and run-directory names are excluded from scientific equality. A second short GPU 1 fixture verifies real CUDA optimizer recovery before the long GPU job.

In [ ]:
def go2_scientific_training_state(state):
    def cpu(value):
        if isinstance(value,torch.Tensor):return value.detach().cpu().clone()
        if isinstance(value,dict):return {k:cpu(v) for k,v in value.items()}
        if isinstance(value,list):return [cpu(v) for v in value]
        if isinstance(value,tuple):return tuple(cpu(v) for v in value)
        return copy.deepcopy(value)
    value=cpu(state)
    value.pop('active_seconds',None)
    value['config'].pop('run_id',None);value['identity']['config'].pop('run_id',None)
    value['metrics'].pop('first_complete_game_seconds',None)
    if value['search'] is not None:
        value['search'].pop('elapsed_seconds',None);value['search'].pop('payload_sha256',None)
    return value

def go2_training_resume_fixture(project,config,device='cpu'):
    import subprocess
    project=Path(project);token=uuid.uuid4().hex[:9]
    full={**copy.deepcopy(config),'run_id':'ahr-v2-training-smoke-'+token+'-full','device':device}
    resumed={**copy.deepcopy(config),'run_id':'ahr-v2-training-smoke-'+token+'-resumed','device':device}
    baseline=go2_run_training(project,full)
    partial=go2_run_training(project,resumed,max_events=2)
    if partial['search'] is None:raise AssertionError('Fixture must pause inside an active search')
    fixture=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','display_name':'AHR Go','language':'python'}})
    fixture.cells=[nbformat.v4.new_code_cell('from pathlib import Path\nimport nbformat\nPROJECT=Path('+repr(str(project))+')\nfor name in '+repr(list(GO2_NOTEBOOKS)+[GO2_TRAINING_NOTEBOOK])+':\n    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:\n        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]):exec(cell.source,globals())'),
        nbformat.v4.new_code_cell('config='+repr(resumed)+'\nstate=go2_run_training(PROJECT,config)\nassert state["metrics"]["optimizer_step"]>0')]
    path=project/'reports/ahr_v2/fixtures'/('training-resume-'+token+'.ipynb');path.parent.mkdir(parents=True,exist_ok=True);nbformat.write(fixture,path)
    subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
        '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=240',str(path)],check=True,capture_output=True,text=True)
    after,_=go2_load_checkpoint(project/'checkpoints'/resumed['run_id'])
    left=go2_scientific_training_state(baseline);right=go2_scientific_training_state(after)
    mismatches=[key for key in left if not go2_nested_equal(left[key],right[key])]
    if mismatches:raise AssertionError('Scientific resume differs: '+str(mismatches))
    model,metadata=go2_load_player(project,resumed['run_id'],device=device)
    output=go2_eval_batch(model,[go2_initial_state(config['board_size'],config['komi'])])[0]
    assert np.isclose(sum(output['prior']),1) and all(np.isfinite(output['prior']))
    assert baseline['metrics']['completed_games']>=1 and baseline['metrics']['optimizer_step']>=1
    return {'device':device,'status':'PASS','full_run':full['run_id'],'resumed_run':resumed['run_id'],
        'completed_games':baseline['metrics']['completed_games'],'unknown_games':baseline['metrics']['unknown_games'],
        'optimizer_steps':baseline['metrics']['optimizer_step'],'event_steps':baseline['global_step'],
        'fresh_kernel_exact_scientific_state':True,'paused_active_search':True,'player_loader':True,
        'first_game_seconds':baseline['metrics']['first_complete_game_seconds'],'fixture_notebook':str(path.relative_to(project))}

def go2_validate_training(project=None):
    project=go2_project(project)
    config=go2_training_config(device='cpu',time_budget_hours=1)
    config.update(board_size=3,komi=0.5,network={'width':8,'blocks':1},search_schedule=[[0,4]],search_batch_size=2,
        max_plies=100,max_total_games=2,max_completed_games=None,batch_size=4,updates_per_game=3,unknown_updates=1,
        replay_train_games=4,replay_dev_games=4,replay_lockbox_games=4,cpu_threads=2,is_smoke=True)
    cpu=go2_training_resume_fixture(project,config,'cpu')
    results=[cpu]
    if torch.cuda.is_available() and torch.cuda.device_count()>1:
        gpu={**config,'max_total_games':1};results.append(go2_training_resume_fixture(project,gpu,'cuda:1'))
    # Inject an optimizer-stage failure after sampler advancement, then recover the exact trajectory.
    global go2_joint_loss
    correct_loss=go2_joint_loss;failed_config={**config,'run_id':'ahr-v2-error-smoke-'+uuid.uuid4().hex[:9]}
    def injected_loss(*args,**kwargs):raise RuntimeError('deliberate_disposable_optimizer_failure')
    go2_joint_loss=injected_loss
    try:
        try:go2_run_training(project,failed_config)
        except RuntimeError as error:
            if 'deliberate_disposable_optimizer_failure' not in str(error):raise
        else:raise AssertionError('Injected failure was ignored')
    finally:go2_joint_loss=correct_loss
    safe,safe_entry=go2_load_checkpoint(project/'checkpoints'/failed_config['run_id'])
    failed_progress=json.loads((project/'checkpoints'/failed_config['run_id']/'progress.json').read_text())
    assert failed_progress['status']=='ERROR' and failed_progress['event_step']==safe['global_step']
    assert safe['metrics']['optimizer_step']==0 and safe['sampler']['rows_seen']==0
    recovered=go2_run_training(project,failed_config)
    baseline,_=go2_load_checkpoint(project/'checkpoints'/cpu['full_run'])
    assert go2_nested_equal(go2_scientific_training_state(baseline),go2_scientific_training_state(recovered))
    # Explicit UNKNOWN policy-only and augmentation checks use the genuine search-game rows.
    saved,_=go2_load_checkpoint(project/'checkpoints'/cpu['full_run'])
    game=saved['replay']['train'][0];unknown=copy.deepcopy(game);unknown['status']='UNKNOWN';unknown['actions']=unknown['actions'][:1]
    unknown['positions']=unknown['positions'][:1];unknown['game_id']+='-unknown-fixture'
    final=go2_replay(go2_initial_state(game['size'],game['komi']),unknown['actions']);unknown['final_state_sha256']=go2_state_digest(final)
    row=go2_replay_row(unknown,0,0,{})
    assert set(row['labels'])=={'policy'}
    terminal=go2_game_cache(game,{})[1]
    original=go2_replay_row(game,0,0,{})
    oracle=go2_training_row(go2_initial_state(game['size'],game['komi']),terminal,
        policy_target=game['positions'][0]['policy'],policy_provenance=original['labels']['policy']['provenance'])
    for kind in ('outcome','score','ownership'):assert np.array_equal(original['labels'][kind]['target'],oracle['labels'][kind]['target'])
    for symmetry in range(8):
        transformed=go2_replay_row(game,0,symmetry,{})
        state=go2_state_from_dict(transformed['state'])
        assert not np.any(np.asarray(transformed['labels']['policy']['target'])[~np.asarray(go2_legal_mask(state),dtype=bool)])
    production_shape=None
    if torch.cuda.is_available() and torch.cuda.device_count()>1:
        large=go2_training_config(run_id='ahr-v2-9x9-learning-smoke-'+uuid.uuid4().hex[:9],time_budget_hours=0.2)
        large.update(is_smoke=True,max_total_games=1)
        started=time.monotonic();large_state=go2_run_training(project,large)
        if large_state['metrics']['completed_games']!=1 or large_state['metrics']['optimizer_step']!=32:
            raise AssertionError('Production-shape fixture did not complete one game and 32 real optimizer updates')
        production_shape={'status':'PASS','run_id':large['run_id'],'device':'cuda:1','board_size':9,
            'network':large['network'],'completed_games':1,'optimizer_steps':32,'unknown_games':large_state['metrics']['unknown_games'],
            'game_plies':large_state['metrics']['recent_games'][0]['plies'],
            'first_game_seconds':large_state['metrics']['first_complete_game_seconds'],
            'active_seconds':large_state['active_seconds'],'wall_seconds':time.monotonic()-started,
            'final_loss':large_state['metrics']['last_loss'],'is_smoke':True}
    receipt={'status':'PASS','sources':go2_training_sources(project),'fixtures':results,'production_shape':production_shape,
        'checks':{'real_complete_game_and_optimizer_learning':True,'fresh_kernel_exact_resume':True,
            'active_game_and_search_restored':True,'unknown_policy_only':True,'terminal_labels_match_02_oracle':True,
            'all_D4_policy_legality':True,'split_before_augmentation':True,'dev_lockbox_never_sampled':True,'partial_optimizer_event_never_checkpointed':True,'error_resume_exact_trajectory':True},
        'scope':'Disposable checks only; 12-hour production run is launched separately by 07','validated_at_utc':go2_now()}
    go2_atomic_json(project/'reports/ahr_v2/training_validation.json',receipt)
    return receipt


In [ ]:
training_validation=go2_validate_training(PROJECT)
from IPython.display import display,HTML
rows=''.join('<tr><td>'+f['device']+'</td><td>'+str(f['completed_games'])+'</td><td>'+str(f['optimizer_steps'])+'</td><td>'+f['status']+'</td></tr>' for f in training_validation['fixtures'])
display(HTML('<h3>Recovery validation passed</h3><table><tr><th>Device</th><th>Complete games</th><th>Optimizer updates</th><th>Fresh-kernel recovery</th></tr>'+rows+'</table>'))
print('UNKNOWN retains policy only. Terminal labels match notebook 02. All eight D4 transforms retain legal policy targets.')
if training_validation['production_shape']:
    proof=training_validation['production_shape']
    print('9×9 GPU 1 fixture:',proof['game_plies'],'plies;',proof['optimizer_steps'],'real updates;',round(proof['wall_seconds'],1),'seconds. Weights remain disposable.')

## Launch, pause and exact resume

Notebook **07** validates these receipts and launches the 12-hour job in tmux. Notebook **05** reports progress and independent GNU Go development matches; notebook **06** opens a clickable board and freezes one saved player snapshot per game. None of these needs to replay training cells to import definitions.

To resume exactly, import the tagged cells of 00, 01, 02, 03 and 04, read `checkpoints/<run_id>/config.json`, then call `go2_run_training(PROJECT, config)`. The latest manifest checkpoint is verified and the saved event/update counters restored. Source/config mismatch or retained payload corruption raises an error; no silent restart.

To stop safely, create the data marker `checkpoints/<run_id>/STOP`. The loop saves the next bounded event and returns `STOPPED`. Remove this marker before an explicitly requested resume. A failed partially completed event is never checkpointed: the ERROR report names the last valid manifest cursor and exception. An explicit resume replays from its saved RNG and state. A process interruption resumes from the latest interruption-safe manifest; at most the configured checkpoint interval is repeated. One writer owns the run lock.

Checkpoints retain latest two valid states and best training-loss state. Each contains all model/optimizer/scheduler/scaler state, global Python/NumPy/Torch and used CUDA RNG, private sampler/action/search RNG, replay, current game, current tree, data position, configuration and metrics. Off-instance backup remains deferred under the user's earlier instruction; these artifacts remain on the persistent boot disk.